In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import MaxNLocator, PercentFormatter

from study.data import cohort, features
from study.features import DYNAMIC_VARS
from study.io import ROOT, WORK, load, reset_directory
from study.models import calibration_curve_frame
from study.plotting import (
    INK,
    LIGHT_GREY,
    MID_GREY,
    NAVY,
    ORANGE,
    PLUM,
    TEAL,
    TERRACOTTA,
    WHITE,
    clean_axis,
    panel_title,
    save_figure,
    style,
)

In [ ]:
COHORT_LABELS = {'carevue': 'CareVue', 'metavision': 'MetaVision', 'mimiciv': 'MIMIC-IV', 'eicu': 'eICU'}

COHORT_STYLES = {
    'carevue': (NAVY, 'o'),
    'metavision': (TEAL, 's'),
    'mimiciv': (ORANGE, '^'),
    'eicu': (TERRACOTTA, 'D')
}

TIER_STYLES = {'R1': (ORANGE, '^'), 'R2': (NAVY, 'o'), 'R3': (TEAL, 's'), 'R4': (PLUM, 'D')}

CI_STYLE = {'elinewidth': 1.1, 'capsize': 0, 'markersize': 5.5, 'markeredgewidth': 0}

LEGEND_STYLE = {
    'loc': 'lower left',
    'bbox_to_anchor': (0, 1.0),
    'borderaxespad': 0,
    'borderpad': 0,
    'labelspacing': 0.3,
    'handlelength': 1.1,
    'handletextpad': 0.4,
    'columnspacing': 1.1
}

SETTINGS = [
    ('carevue', 'carevue', 'original'),
    ('carevue', 'metavision', 'original'),
    ('metavision', 'mimiciv', 'shared'),
    ('metavision', 'eicu', 'shared')
]

LOCAL_CONDITIONS = ['full', 'retain_0.5', 'retain_0.25', 'terminal_outage']
CALIBRATION_CONDITIONS = ['full', 'retain_0.25', 'terminal_outage']

In [ ]:
reset_directory(ROOT / 'results/figures')

# Check the source design and complete-cohort comparisons
source_fits = load(WORK / 'models/source.pkl')

full = pd.read_parquet(WORK / 'predictions/full')
physiology = pd.read_parquet(WORK / 'results/physiology.parquet')
observation = pd.read_parquet(WORK / 'results/observation.parquet')

timing = pd.read_parquet(WORK / 'results/timing.parquet')
local = pd.read_parquet(WORK / 'results/local_availability.parquet')
local_calibration = pd.read_parquet(WORK / 'results/local_calibration.parquet')

style()

In [ ]:
# Count the full cohorts for the study diagram
rows = []
for domain in ('carevue', 'metavision', 'mimiciv', 'eicu'):
    metadata = cohort(domain)
    rows.append(dict(
        domain = domain, partition = 'full', patients = len(metadata),
        hospitals = metadata.hospital_id.nunique() if domain == 'eicu' else 1,
    ))

cohorts = pd.DataFrame(rows)

In [ ]:
# Prepare raw shared feature calibration curves
curves = []
selected = full.loc[full.source.eq('metavision') & full.support.eq('shared') & full.tier.isin(['R1', 'R2', 'R3'])]
for (domain, tier, repeat), group in selected.groupby(['domain', 'tier', 'repeat'], observed = True):
    group = group.loc[group.partition.eq('test')] if domain == 'eicu' else group
    curve = calibration_curve_frame(group.in_hospital_mortality, group.risk)
    curves.append(curve.assign(domain = domain, tier = tier, repeat = repeat))

calibration_curves = (
    pd.concat(curves, ignore_index = True)
    .groupby(['domain', 'tier', 'bin'], observed = True)[['mean_predicted_mortality', 'observed_mortality']]
    .mean()
    .reset_index()
)

# Select hospitals using the reported thresholds
sites = pd.read_parquet(WORK / 'results/hospital_effects.parquet')
eligible = sites.loc[sites.patients.ge(100) & sites.deaths.ge(20) & (sites.patients - sites.deaths).ge(20)]

In [ ]:
# Information increments
labels = [
    'CareVue (Internal)',
    'CareVue → MetaVision',
    'MetaVision (Internal)',
    'MetaVision → MIMIC-IV',
    'MetaVision → eICU',
    'eICU (Local)'
]

def row(table, **conditions):
    mask = pd.Series(True, index = table.index)
    for column, value in conditions.items():
        mask &= table[column].eq(value)
    selected = table.loc[mask]
    if len(selected) != 1:
        raise ValueError(f'Expected one row for {conditions}, found {len(selected)}')
    return selected.iloc[0]

fig, axes = plt.subplots(1, 2, figsize = (11.8, 4.9), sharex = True, sharey = True)

for axis, frame, contrast, tier, title in zip(
    axes,
    (physiology, observation),
    ('R2_minus_R1', 'R3_minus_R2'),
    ('R2', 'R3'),
    ['A: Physiological Trajectory (R2–R1)', 'B: Observation Patterns (R3−R2)'],
):
    rows = [
        row(frame, source = 'carevue', domain = 'carevue', support = 'original', weighting = 'patient'),
        row(frame, source = 'carevue', domain = 'metavision', support = 'original', weighting = 'patient'),
        row(frame, source = 'metavision', domain = 'metavision', support = 'shared', weighting = 'patient'),
        row(frame, source = 'metavision', domain = 'mimiciv', support = 'shared', weighting = 'patient'),
        row(frame, source = 'metavision', domain = 'eicu', support = 'shared', weighting = 'patient'),
        row(local, training = 'full', restriction = 'full', contrast = contrast, weighting = 'patient')
    ]

    data = pd.DataFrame(rows, index = labels)
    positions = np.arange(len(labels))
    colour, marker = TIER_STYLES[tier]

    axis.errorbar(
        data.difference.to_numpy(),
        positions,
        xerr = [
            (data.difference - data.lower).to_numpy(),
            (data.upper - data.difference).to_numpy()
        ],
        fmt = marker,
        color = colour,
        **CI_STYLE,
    )

    axis.axvline(0, color = INK, lw = 0.85)
    axis.set_yticks(positions, labels)
    axis.set(ylim = (len(labels) - 0.5, -0.5), xlabel = 'Paired log-loss difference')
    axis.xaxis.set_major_locator(MaxNLocator(5))
    
    panel_title(axis, title)

    clean_axis(axis, 'x')

fig.tight_layout(pad = 1.2, w_pad = 3)

save_figure(fig, 'information_increments')

In [ ]:
# Count matched timing effects in each tier
selected = timing.loc[
    timing.domain.eq('eicu')
    & timing.source.eq('metavision')
    & timing.support.eq('shared')
    & timing.weighting.eq('patient')
]

fig, axes = plt.subplots(1, 3, figsize = (10, 3.8), sharey = True)

for axis, (contrast, group), title in zip(
    axes,
    selected.groupby('contrast'),
    ['A: Recent vs Random', 'B: Spread vs Random', 'C: Outage vs Matched Random'],
):
    group = group.sort_values('tier')

    for position, row in enumerate(group.itertuples()):
        colour, _ = TIER_STYLES[row.tier]
        axis.errorbar(
            position,
            row.difference,
            yerr = [[row.difference - row.lower], [row.upper - row.difference]],
            fmt = 'o',
            color = colour,
            **CI_STYLE,
        )

    axis.axhline(0, color = INK, lw = 0.85)
    axis.set_xticks(range(4), group.tier)
    axis.set(xlim = (-0.5, 3.5), xlabel = 'Representation')
    
    panel_title(axis, title)
    
    clean_axis(axis, 'y')

axes[0].set_ylabel('Paired log-loss difference')

fig.tight_layout(pad = 1.2, w_pad = 2.5)

save_figure(fig, 'timing_decomposition')

In [ ]:
# Local information increments and matched calibration
fig, axes = plt.subplots(1, 2, figsize = (10, 4.4))

for contrast, tier in [('R2_minus_R1', 'R2'), ('R3_minus_R2', 'R3')]:
    data = (
        local.loc[local.training.eq('full') & local.weighting.eq('patient') & local.contrast.eq(contrast)]
        .set_index('restriction')
        .loc[LOCAL_CONDITIONS]
    )

    colour, marker = TIER_STYLES[tier]
    axes[0].errorbar(
        range(len(LOCAL_CONDITIONS)),
        data.difference,
        yerr = [data.difference - data.lower, data.upper - data.difference],
        fmt = marker + '-',
        color = colour,
        label = contrast.replace('_minus_', '−'),
        **CI_STYLE,
    )

for name, colour, marker in [('raw', NAVY, 'o'), ('matched', TEAL, 's')]:
    data = (
        local_calibration.loc[
            local_calibration.weighting.eq('patient')
            & (
                local_calibration.calibration.eq('raw')
                if name == 'raw'
                else local_calibration.calibration.str.endswith(':platt')
            )
        ]
        .set_index('restriction')
        .loc[CALIBRATION_CONDITIONS]
    )

    axes[1].errorbar(
        range(len(CALIBRATION_CONDITIONS)),
        data.difference,
        yerr = [data.difference - data.lower, data.upper - data.difference],
        fmt = marker + '-',
        color = colour,
        label = 'Raw' if name == 'raw' else 'Matched calibration',
        **CI_STYLE,
    )

for axis, title in zip(axes, ['A: Local Availability', 'B: Matched Calibration']):
    axis.axhline(0, color = INK, lw = 0.85)
    axis.set(ylabel = 'Paired log-loss difference', xlabel = 'Input availability')
    axis.yaxis.set_major_locator(MaxNLocator(5))
    axis.legend(ncol = 2, **LEGEND_STYLE | {'bbox_to_anchor': (0, 1.025)})
    axis.set_title(title, loc = 'left', pad = 22)
    
    clean_axis(axis, 'y')

axes[0].set_xticks(range(4), ['Full', '50%', '25%', 'Outage'])
axes[1].set_xticks(range(3), ['Full', '25%', 'Outage'])

fig.tight_layout(pad = 1.2, w_pad = 4)

save_figure(fig, 'availability_and_calibration')

In [ ]:
# Recording coverage and median last measurement time
rows = []
for domain in ('carevue', 'metavision', 'mimiciv', 'eicu'):
    metadata, x = (cohort(domain), features(domain))
    ids = metadata.index[metadata.partition.eq('test')] if domain == 'eicu' else metadata.index

    # Summarise recording coverage, counts, and last measurement time
    for variable in [name for name in DYNAMIC_VARS if name != 'MechVent']:
        count = x.loc[ids, variable + '_count']
        observed = count.gt(0)
        rows.append(
            dict(
                domain = domain,
                variable = variable,
                fraction = observed.mean(),
                median_count = count.loc[observed].median(),
                last_hour = x.loc[ids, variable + '_last_measurement_hour'].loc[observed].median(),
            )
        )

recording = pd.DataFrame(rows)
fig, axes = plt.subplots(1, 2, figsize = (10, 8.8), sharey = True)

for offset, (domain, (colour, marker)) in zip(np.linspace(-0.18, 0.18, 4), COHORT_STYLES.items()):
    group = recording.loc[recording.domain.eq(domain)].sort_values('variable')
    positions = np.arange(len(group)) + offset

    for axis, values in zip(axes, (group.fraction, group.last_hour)):
        axis.scatter(
            values,
            positions,
            s = 23,
            color = colour,
            marker = marker,
            edgecolor = WHITE,
            linewidth = 0.35,
            label = COHORT_LABELS[domain],
            zorder = 3,
        )

axes[0].set_yticks(range(len(group)), group.variable)
axes[0].set_ylim(len(group) - 0.5, -0.5)
axes[0].set(xlabel = 'Patients with a recording', xlim = (-0.03, 1.03))
axes[0].set_xticks(np.linspace(0, 1, 6))
axes[0].xaxis.set_major_formatter(PercentFormatter(1, decimals = 0))
axes[0].legend(ncol = 4, **LEGEND_STYLE | {'bbox_to_anchor': (0, 1.01)})
axes[1].set(xlabel = 'Median last recording time (hours)', xlim = (-1, 49))
axes[1].set_xticks([0, 12, 24, 36, 48])

for axis, title in zip(axes, ['A: Recording Coverage', 'B: Last Recording']):
    axis.set_title(title, loc = 'left', pad = 22)
    
    clean_axis(axis, 'x')

fig.tight_layout(pad = 1.2, w_pad = 4)

save_figure(fig, 'observation_description')

In [ ]:
# Raw shared feature decile calibration
fig, axes = plt.subplots(1, 3, figsize = (10, 4), sharex = True, sharey = True)
limit = min(
    1.0,
    np.ceil(calibration_curves[['mean_predicted_mortality', 'observed_mortality']].max().max() * 10) / 10 + 0.05,
)

for axis, domain, letter in zip(axes, ('metavision', 'mimiciv', 'eicu'), 'ABC'):
    for tier, group in calibration_curves.loc[calibration_curves.domain.eq(domain)].groupby('tier'):
        colour, marker = TIER_STYLES[tier]
        axis.plot(
            group.mean_predicted_mortality,
            group.observed_mortality,
            marker = marker,
            color = colour,
            markersize = 4,
            label = tier,
        )

    axis.plot([0, limit], [0, limit], color = MID_GREY, lw = 0.8, ls = (0, (3, 3)), zorder = 1)
    axis.set(xlabel = 'Predicted mortality', xlim = (0, limit), ylim = (0, limit))
    axis.set_aspect('equal', adjustable = 'box')
    axis.xaxis.set_major_formatter(PercentFormatter(1, decimals = 0))
    axis.yaxis.set_major_formatter(PercentFormatter(1, decimals = 0))
    axis.xaxis.set_major_locator(MaxNLocator(4))
    axis.yaxis.set_major_locator(MaxNLocator(4))
    axis.legend(ncol = 3, **LEGEND_STYLE | {'bbox_to_anchor': (0, 1.025)})
    axis.set_title(f'{letter}: {COHORT_LABELS[domain]}', loc = 'left', pad = 22)
    
    clean_axis(axis, 'both')

axes[0].set_ylabel('Observed mortality')

fig.tight_layout(pad = 1.2, w_pad = 3)

save_figure(fig, 'raw_shared_calibration')

In [ ]:
# Hospital-specific information increments
fig, axes = plt.subplots(2, 1, figsize = (10, 9))

for axis, (contrast, group), tier, title in zip(
    axes,
    eligible.groupby('contrast'),
    ('R2', 'R3'),
    ['A: Physiological Trajectory (R2−R1)', 'B: Observation Patterns (R3−R2)'],
):
    group = group.sort_values('difference')
    colour, marker = TIER_STYLES[tier]
    axis.errorbar(
        np.arange(len(group)),
        group.difference,
        yerr = 1.96 * group.se,
        fmt = marker,
        color = colour,
        ecolor = LIGHT_GREY,
        elinewidth = 0.75,
        markersize = 3.2,
        markeredgewidth = 0,
        capsize = 0,
    )

    axis.axhline(0, color = INK, lw = 0.85)
    axis.set(xlabel = 'Hospitals ordered by their own increment', ylabel = 'Paired log-loss difference')
    axis.xaxis.set_major_locator(MaxNLocator(4, integer = True))
    axis.yaxis.set_major_locator(MaxNLocator(5))
    
    panel_title(axis, title)
    
    clean_axis(axis, 'y')

fig.tight_layout(pad = 1.2, h_pad = 3)

save_figure(fig, 'external_hospital_heterogeneity')